# 10 — What the Balancing Mechanism does to every fleet figure

Notebooks 05, 08 and 09 measure the fleet from **Final Physical Notifications**. A PN is
what a unit told the operator it intended to do at gate closure. After gate closure NESO
accepts bids and offers that instruct it away from that plan, and for GB batteries the
accepted volume is of the same order as the notified position — so *every* claim about what
the fleet delivered, held or spent has to be checkable on both bases.

This notebook rebuilds three of the project's headline results on the corrected series and
reports the difference. It changes no other notebook's numbers; it says how far they can be
trusted.

The corrected series is the store's `fleet_boa` table — the notification painted on a minute
grid, then acceptances painted over the top (`fleet.performance.site_physical_profile`), so a
three-minute instruction cannot rewrite a half-hour.

In [1]:
import datetime as dt
import importlib.util
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pulp

# Repo root, found by walking up to the marker file, so the notebook runs the
# same whether it is opened from its own directory or from the repo root.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT))
warnings.filterwarnings("ignore")

from research.figexport import save_poster_metrics
from fleet.research import census
from fleet import performance as fleet_perf
from fleet.population import census_population

_spec = importlib.util.spec_from_file_location(
    "build_stress_store", REPO_ROOT / "scripts" / "build_stress_store.py")
bss = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(bss)

# Same frozen vintage as notebooks 04-09.
census.SNAPSHOT = dt.date(2026, 8, 24)
WIN_START, WIN_END = dt.date(2026, 6, 26), dt.date(2026, 8, 24)
SKIP_END = pd.Timestamp("2023-10-01", tz="UTC")
MODERN_START = pd.Timestamp("2024-04-01", tz="UTC")
LOLP_RULE, ETA_C, ETA_D = 1e-4, 0.94, 0.94
MIN_SOC, MAX_SOC, CYCLES = 0.10, 1.0, 1.5

POP = census_population()
SITE_MW = {s.site: s.power_mw for s in POP.sites}
SITE_MWH = {s.site: s.capacity_mwh for s in POP.sites}
S = bss.load_store(bss.store_for(POP))

pn = S["fleet_pn"].copy(); pn["time"] = pd.to_datetime(pn["time"], utc=True)
boa = S["fleet_boa"].copy(); boa["time"] = pd.to_datetime(boa["time"], utc=True)
pn, boa = pn[pn["site"].isin(SITE_MW)], boa[boa["site"].isin(SITE_MW)]

# The pre-battery correction notebooks 05 and 07 apply, on both bases.
ERA_START = fleet_perf.battery_era_start(pn, SITE_MW)
def _cut(f):
    if not ERA_START:
        return f
    keep = pd.Series(True, index=f.index)
    for site, vf in ERA_START.items():
        keep &= ~((f["site"] == site) & (f["time"] < vf))
    return f[keep]
pn, boa = _cut(pn), _cut(boa)

print(f"Population : {len(SITE_MW)} sites, {sum(SITE_MW.values()):,.0f} MW")
print(f"fleet_pn   : {len(pn):,} rows")
print(f"fleet_boa  : {len(boa):,} rows")

Worksheet row for GB-BESS-WOLVB (Wolverhampton West BESS) needs review: 310 MWh over 56.0 MW declared implies 5.5 h, longer than any priced census site. Self-consistent with the duration recorded, so the agreement check cannot judge it — confirm the figure covers this BM Unit and not the wider project.


Population : 87 sites, 6,234 MW
fleet_pn   : 4,256,779 rows
fleet_boa  : 4,261,717 rows


## 1. Section 3's window — delivery into top-decile load

Notebook 09's yardstick, unchanged: the same 60 days, the same top-decile residual-load bar.
Only the actual-delivery series changes.

In [2]:
resid = S["system"]["residual_mw"].loc[
    pd.Timestamp(WIN_START, tz="UTC"):pd.Timestamp(WIN_END, tz="UTC") + pd.Timedelta(days=1)
].dropna()
thresh = float(resid.quantile(0.90))
stress_hh = resid >= thresh
print(f"Top-decile bar : {thresh/1000:.1f} GW  ({int(stress_hh.sum())} half-hours)")
print("This must match notebook 09's printed 23.3 GW: same function, same window.")
print("Delivered energy is computed in the next cell, on the identical hourly")
print("grid, flag convention and site set as both achievable denominators.")

Top-decile bar : 23.3 GW  (288 half-hours)
This must match notebook 09's printed 23.3 GW: same function, same window.
Delivered energy is computed in the next cell, on the identical hourly
grid, flag convention and site set as both achievable denominators.


### The achievable denominator, so the shares are comparable

Rebuilt here rather than imported, with a **period-by-period** MELS bound: notebook 09 uses
the mean declared limit across the flagged hours, and a flat mean can move energy into hours
a site was not available. The difference is under a point (see notebook 09's sensitivity), but
the point of this notebook is not to inherit an approximation.

In [3]:
mels = S["fleet_mels"].copy(); mels["time"] = pd.to_datetime(mels["time"], utc=True)
W0 = pd.Timestamp(WIN_START, tz="UTC")
W1 = pd.Timestamp(WIN_END, tz="UTC") + pd.Timedelta(days=1)

# Window BOTH frames before pivoting. Pivoting the full 2018-2026 history and
# then filtering per day is what turns a seven-second job into an hours-long
# one: the per-day index scan is O(days x sites x whole-history).
def _hourly(frame):
    f = frame[(frame["time"] >= W0) & (frame["time"] <= W1)]
    return (f.pivot_table(index="time", columns="site", values="mw", aggfunc="sum")
             .resample("1h").mean())

mels_h, pn_h, boa_h = _hourly(mels), _hourly(pn), _hourly(boa)
stress_h = stress_hh.resample("1h").max().astype(bool)
hours = stress_h.index
by_day = {d: hours[hours.date == d] for d in sorted({t.date() for t in hours})}
print(f"Hourly grid: {len(hours):,} hours over {len(by_day)} days")
print(f"Top-decile hours: {int(stress_h.sum())} of {len(hours)}  "
      f"(notebook 09 prints 161 of 1,439)\n")

def achievable(cap, limits, flags):
    n = len(flags)
    prob = pulp.LpProblem("res", pulp.LpMaximize)
    ch = [pulp.LpVariable(f"c{h}", 0, float(limits[h])) for h in range(n)]
    di = [pulp.LpVariable(f"d{h}", 0, float(limits[h])) for h in range(n)]
    so = [pulp.LpVariable(f"s{h}", MIN_SOC*cap, MAX_SOC*cap) for h in range(n+1)]
    prob += pulp.lpSum(di[h]*(2.0 if flags[h] else -2.0) + ch[h]*(-2.0 if flags[h] else -1e-3)
                       for h in range(n))
    prob += so[0] == cap*0.5
    for h in range(n):
        prob += so[h+1] == so[h] - di[h]/ETA_D + ch[h]*ETA_C
    prob += pulp.lpSum(di) <= CYCLES*cap
    prob.solve(_SOLVER)
    return sum(di[h].value() or 0.0 for h in range(n) if flags[h])

try:
    import highspy  # noqa: F401
    _SOLVER = pulp.HiGHS(msg=0)
except ImportError:
    _SOLVER = pulp.PULP_CBC_CMD(msg=0)

sites = [s for s in SITE_MWH
         if SITE_MWH.get(s) == SITE_MWH.get(s) and SITE_MWH.get(s, 0) > 0
         and s in pn_h.columns and s in mels_h.columns]
print(f"Measurable sites (MWh known, reporting in window): {len(sites)}")

# Delivered energy on the identical convention as the denominators: the same
# sites, the same hourly grid, the same flagged hours, positive output only.
flag_hours = hours[stress_h.to_numpy()]
deliv = {}
for name, hp in (("pn", pn_h), ("boa", boa_h)):
    cols = [s for s in sites if s in hp.columns]
    deliv[name] = float(hp.loc[hp.index.isin(flag_hours), cols]
                        .clip(lower=0).sum().sum())
cut = 1 - deliv["boa"] / deliv["pn"]
print(f"Delivered, notified schedule           : {deliv['pn']:>10,.0f} MWh")
print(f"  (notebook 09, same convention, prints 121,756 MWh)")
print(f"Delivered, acceptance-adjusted profile : {deliv['boa']:>10,.0f} MWh")
print(f"Reduction from netting acceptances     : {cut:.1%}")

ach_name = ach_mels = 0.0
for site in sites:
    cap = SITE_MWH[site]
    for day, idx in by_day.items():
        if len(idx) < 20:
            continue
        fl = stress_h.reindex(idx).fillna(False).to_numpy()
        if not fl.any() or pn_h[site].reindex(idx).isna().all():
            continue
        ach_name += achievable(cap, np.full(len(idx), SITE_MW[site]), fl)
        dec = np.clip(mels_h[site].reindex(idx).fillna(0.0).to_numpy(), 0, None)
        if dec[fl].max() > 0:
            ach_mels += achievable(cap, dec, fl)

print(f"\nAchievable vs nameplate : {ach_name:>10,.0f} MWh")
print(f"Achievable vs declared  : {ach_mels:>10,.0f} MWh\n")
for base, val in (("notifications", deliv["pn"]), ("+ acceptances", deliv["boa"])):
    print(f"  {base:<15} vs nameplate {val/ach_name:5.0%}   vs declared {val/ach_mels:5.0%}")
pts = (deliv["pn"] - deliv["boa"]) / ach_mels
print(f"\nAcceptances cost {pts:.0%} of the declared-availability share.")

Hourly grid: 1,439 hours over 60 days
Top-decile hours: 161 of 1439  (notebook 09 prints 161 of 1,439)

Measurable sites (MWh known, reporting in window): 63
Delivered, notified schedule           :    121,756 MWh
  (notebook 09, same convention, prints 121,756 MWh)
Delivered, acceptance-adjusted profile :     89,363 MWh
Reduction from netting acceptances     : 26.6%



Achievable vs nameplate :    289,102 MWh
Achievable vs declared  :    152,032 MWh

  notifications   vs nameplate   42%   vs declared   80%
  + acceptances   vs nameplate   31%   vs declared   59%

Acceptances cost 21% of the declared-availability share.


## 2. Section 4's lane — response under operator scarcity

The same absolute rule notebooks 05 and 08 use: LoLP ≥ 1e-4, normalised per MW online.

In [4]:
prints = S["lolpdrm_prints"]
final = (prints.sort_values(["horizon", "publish_time"], ascending=[True, False])
               .drop_duplicates("time").set_index("time")[["lolp"]].sort_index())
tight_idx = final.index[final["lolp"] >= LOLP_RULE]

pn["date"] = pn["time"].dt.date
span = pn.groupby("site")["date"].agg(["min", "max"])
all_days = pd.DatetimeIndex(sorted(pn["date"].unique()), tz="UTC")
online = pd.Series(0.0, index=all_days)
for site, row in span.iterrows():
    live = (all_days.date >= row["min"]) & (all_days.date <= row["max"])
    online[live] += SITE_MW.get(site, 0.0)

resp = {}
for name, frame in (("pn", pn), ("boa", boa)):
    net = frame[frame["time"].isin(tight_idx)].groupby("time")["mw"].sum()
    on = online.reindex(net.index.normalize()).to_numpy()
    resp[name] = pd.Series(net.to_numpy() / np.where(on > 0, on, np.nan), index=net.index).dropna()
wide = pd.DataFrame(resp).dropna()
pre, mod = wide[wide.index < SKIP_END], wide[wide.index >= MODERN_START]
print(f"Scarcity half-hours on both bases : {len(wide):,}\n")
for lbl, sub in (("Full window", wide), ("Pre-break", pre), ("Modern era", mod)):
    print(f"  {lbl:<12} n={len(sub):>5}  PN {sub['pn'].mean():+.3f}  "
          f"BOA {sub['boa'].mean():+.3f}  ({sub['boa'].mean()-sub['pn'].mean():+.3f})")
print(f"\n  instructed up {(wide['boa']>wide['pn']).mean():.0%} | "
      f"down {(wide['boa']<wide['pn']).mean():.0%} | "
      f"unchanged {(wide['boa']==wide['pn']).mean():.0%}")
print(f"  era ratio  notifications {mod['pn'].mean()/pre['pn'].mean():.2f}x   "
      f"acceptances {mod['boa'].mean()/pre['boa'].mean():.2f}x")

Scarcity half-hours on both bases : 2,063

  Full window  n= 2063  PN +0.060  BOA +0.056  (-0.004)
  Pre-break    n= 1792  PN +0.047  BOA +0.047  (-0.001)
  Modern era   n=  230  PN +0.121  BOA +0.096  (-0.024)

  instructed up 26% | down 31% | unchanged 43%
  era ratio  notifications 2.55x   acceptances 2.07x


## 3. Readiness — the finding the whole board rests on

Charge at onset and energy still held at the deepest point, integrated across the full window
on both bases. If acceptances moved these, notebook 05's and 08's readiness story would be an
artefact of measuring plans rather than delivery.

In [5]:
grid = pd.DatetimeIndex(sorted(set(pn["time"]) | set(boa["time"]))).sort_values()
tight = (final["lolp"] >= LOLP_RULE).reindex(grid).fillna(False)

def events(mask, bridge=2, min_len=2):
    m = mask.to_numpy().copy(); idx = np.flatnonzero(m)
    if idx.size == 0: return []
    for a, b in zip(idx[:-1], idx[1:]):
        if 1 < b - a <= bridge + 1: m[a:b] = True
    ev, st = [], None
    for i, v in enumerate(m):
        if v and st is None: st = i
        elif not v and st is not None:
            if i - st >= min_len: ev.append((st, i))
            st = None
    if st is not None and len(m) - st >= min_len: ev.append((st, len(m)))
    return ev

ev = events(tight)
print(f"Events: {len(ev)}\n")
soc_of = {}
for name, frame in (("pn", pn), ("boa", boa)):
    out = fleet_perf.fleet_state_of_charge(frame, grid, SITE_MWH, SITE_MW, ETA_C, ETA_D)
    soc_of[name] = out["soc"].reindex(grid)
    print(f"  {name}: {len(out['usable'])} usable sites, {len(out['skipped_no_mwh'])} without MWh")

read = {}
for name, soc in soc_of.items():
    for lbl, sub in (("full", ev), ("modern", [e for e in ev if grid[e[0]] >= MODERN_START])):
        on = [soc.iloc[a] for a, _ in sub if not np.isnan(soc.iloc[a])]
        dp = [np.nanmin(soc.iloc[a:b].to_numpy()) for a, b in sub
              if not np.all(np.isnan(soc.iloc[a:b].to_numpy()))]
        read[(name, lbl)] = (float(np.median(on)), float(np.median(dp)))
print("\nCharge at onset (median across events):")
for lbl in ("full", "modern"):
    a, b = read[("pn", lbl)][0], read[("boa", lbl)][0]
    print(f"  {lbl:<8} PN {a:.0%} -> acceptances {b:.0%}  ({(b-a)*100:+.1f} pts)")
print("Energy still held at the deepest point:")
for lbl in ("full", "modern"):
    a, b = read[("pn", lbl)][1], read[("boa", lbl)][1]
    print(f"  {lbl:<8} PN {a:.0%} -> acceptances {b:.0%}  ({(b-a)*100:+.1f} pts)")

Events: 345



  pn: 45 usable sites, 20 without MWh


  boa: 45 usable sites, 20 without MWh

Charge at onset (median across events):
  full     PN 61% -> acceptances 63%  (+2.4 pts)
  modern   PN 45% -> acceptances 47%  (+1.7 pts)
Energy still held at the deepest point:
  full     PN 45% -> acceptances 48%  (+2.6 pts)
  modern   PN 31% -> acceptances 32%  (+0.8 pts)


### Why the onset figures here differ from notebook 08's

Notebook 08 reports a median onset charge of **47%** for the modern era; this notebook
computes **45%** on the same data. Neither is wrong — they bracket different event sets:

* Notebook 08 bridges events *within* the modern era, so an episode straddling April 2024 is
  cut at the boundary and only its later half is scored.
* This notebook bridges across the whole 2018-2026 window and then selects events that
  *start* in the modern era, so those straddling episodes are scored whole.

The gap is the handful of boundary events, and it is smaller than the re-anchoring
sensitivity either notebook already carries. Quote notebook 08's figure for the modern era —
it is the notebook of record for that window. What this notebook adds is the *difference
between bases*, which is computed on one event set and is therefore unaffected.

## 4. What this settles

**Section 3 moves, and the direction is the finding.** In top-decile *load* hours the operator
instructs batteries **down**, not up, so delivery measured on notifications overstates what
arrived. Part of the distance between declared capability and delivered energy is post-gate
instruction, not battery choice — and an availability obligation aimed only at owners would
miss that share.

**Section 4 does not move.** Under genuine scarcity the instruction effect is close to
balanced and the readiness figures shift by about two points. The 61% → 47% decline in charge
at onset is a property of the fleet, not of the measurement.

**Read together:** the operator reshapes battery output at the evening peak and barely touches
it when short. That asymmetry is why the two lanes cannot share a correction, any more than
they can share a threshold.

In [6]:
nb10_metrics = {
    "window": f"{WIN_START} \u2192 {WIN_END}",
    # Section 3 — the correction that matters.
    "cut_pct": f"{cut:.0%}",
    "delivered_pn": f"{deliv['pn']:,.0f} MWh",
    "delivered_boa": f"{deliv['boa']:,.0f} MWh",
    "points_vs_declared": f"{pts:.0%}",
    "share_nameplate_pn": f"{deliv['pn']/ach_name:.0%}",
    "share_nameplate_boa": f"{deliv['boa']/ach_name:.0%}",
    "share_declared_pn": f"{deliv['pn']/ach_mels:.0%}",
    "share_declared_boa": f"{deliv['boa']/ach_mels:.0%}",
    # The cascade: one harness, four gates, each with a different owner.
    "ach_nameplate": f"{ach_name:,.0f} MWh",
    "ach_declared": f"{ach_mels:,.0f} MWh",
    "gate_declared": f"{ach_mels/ach_name:.0%}",
    "gate_planned": f"{deliv['pn']/ach_mels:.0%}",
    "gate_delivered": f"{deliv['boa']/deliv['pn']:.0%}",
    "gate_instructed_away": f"{1 - deliv['boa']/deliv['pn']:.0%}",
    # Section 4 — the correction that does not.
    "response_pn": f"{wide['pn'].mean():+.3f}",
    "response_boa": f"{wide['boa'].mean():+.3f}",
    "modern_pn": f"{mod['pn'].mean():+.3f}",
    "modern_boa": f"{mod['boa'].mean():+.3f}",
    "ratio_pn": f"{mod['pn'].mean()/pre['pn'].mean():.1f}x",
    "ratio_boa": f"{mod['boa'].mean()/pre['boa'].mean():.1f}x",
    "up_share": f"{(wide['boa']>wide['pn']).mean():.0%}",
    "down_share": f"{(wide['boa']<wide['pn']).mean():.0%}",
    "n_scarcity": f"{len(wide):,}",
    "onset_pn": f"{read[('pn','full')][0]:.0%}",
    "onset_boa": f"{read[('boa','full')][0]:.0%}",
    "onset_modern_pn": f"{read[('pn','modern')][0]:.0%}",
    "onset_modern_boa": f"{read[('boa','modern')][0]:.0%}",
    "events": f"{len(ev)}",
}
import json
print(json.dumps(nb10_metrics, indent=2))
save_poster_metrics("nb10", nb10_metrics)

{
  "window": "2026-06-26 \u2192 2026-08-24",
  "cut_pct": "27%",
  "delivered_pn": "121,756 MWh",
  "delivered_boa": "89,363 MWh",
  "points_vs_declared": "21%",
  "share_nameplate_pn": "42%",
  "share_nameplate_boa": "31%",
  "share_declared_pn": "80%",
  "share_declared_boa": "59%",
  "ach_nameplate": "289,102 MWh",
  "ach_declared": "152,032 MWh",
  "gate_declared": "53%",
  "gate_planned": "80%",
  "gate_delivered": "73%",
  "gate_instructed_away": "27%",
  "response_pn": "+0.060",
  "response_boa": "+0.056",
  "modern_pn": "+0.121",
  "modern_boa": "+0.096",
  "ratio_pn": "2.5x",
  "ratio_boa": "2.1x",
  "up_share": "26%",
  "down_share": "31%",
  "n_scarcity": "2,063",
  "onset_pn": "61%",
  "onset_boa": "63%",
  "onset_modern_pn": "45%",
  "onset_modern_boa": "47%",
  "events": "345"
}


PosixPath('/Users/abhinav/Documents/code/power-trading/reports/figures/poster/nb10_metrics.json')